## Query public IP and generate ssh key

We curl the public IP address of our container and also generate a temporary throw-away key for the bastion host

In [ ]:
import os
import subprocess


ip = subprocess.check_output(["curl", "-s", "https://ipinfo.io/ip"]).decode().strip()

print("=== PUBLIC IP ADDRESS (add this to ufw allow on your server) ===\n")
print(f"ufw allow from {ip} to any port 22 proto tcp\n")

print("=== CLEANUP commands (after the exercise) ===\n")
print(f'''x="$(ufw status numbered | grep "{ip}" | awk -F'[][]' '{{print $2}}')" ; ufw delete $x\n''')

# Paths
ssh_dir = "/root/.ssh"
key_name = "colab_temp_key"
key_path = os.path.join(ssh_dir, key_name)
# Ensure ~/.ssh exists
os.makedirs(ssh_dir, exist_ok=True)
os.chmod(ssh_dir, 0o700)
# Delete existing key (private + public) if present
for path in [key_path, key_path + ".pub"]:
    if os.path.exists(path):
        os.remove(path)

# 1. Generate SSH key (ed25519, no passphrase)
subprocess.run([
    "ssh-keygen",
    "-t", "ed25519",
    "-f", key_path,
    "-N", "",  # no passphrase
    "-q"       # quiet
], check=True)


# 2. Set correct permissions
os.chmod(key_path, 0o600)          # private key
os.chmod(key_path + ".pub", 0o644) # public key

# 3. Print public key
with open(key_path + ".pub", "r") as f:
    pubkey = f.read()

print("=== PUBLIC KEY (add this to authorized_keys on your server) ===\n")
print(pubkey)

pubkey=pubkey.strip()  # remove any trailing newlines
authorized_keys_file = "~/.ssh/authorized_keys"

# ---- Generate Bash command to append the key ----
# Escape single quotes in the key for safe echo
escaped_key_for_echo = pubkey.replace("'", "'\"'\"'")

add_key_cmd = f"echo '{escaped_key_for_echo}' >> {authorized_keys_file}"
print("=== BASH COMMAND TO ADD PUBLIC KEY ===")
print(add_key_cmd, "\n")

# ---- Generate Bash command to remove the key ----
# Escape / and & for sed
escaped_key_for_sed = pubkey.replace("/", "\\/").replace("&", "\\&")

remove_key_cmd = f"sed -i '/{escaped_key_for_sed}/d' {authorized_keys_file}"
print("=== BASH COMMAND TO REMOVE PUBLIC KEY ===")
print(remove_key_cmd)


# Generate reverse ssh Tunnel

We want to create a reverse ssh tunnel to our bastion host. For this we need to add HOSTNAME and PRIVATE_KEY to the user data secrets. They will then be read 

In [ ]:
import os
import subprocess
from google.colab import userdata
import getpass

# Get hostname from user
hostname = input("Enter hostname (user@host): ")

# Define local and remote ports
LOCALPORT = 8188 #@param {type:"integer"}
REMOTEPORT = 8100 #@param {type:"integer"}

if not hostname:
    raise ValueError("Missing HOSTNAME or PRIVATE_KEY secret")

# the ssh key which has been generated in the previous cell
key_path = "/root/.ssh/colab_temp_key"

# Build SSH command (reverse tunnel: remote e.g. 8100 -> local e.g. 8199)
ssh_command = f"""
ssh -i {key_path} \
    -o StrictHostKeyChecking=no \
    -o UserKnownHostsFile=/dev/null \
    -N -R {REMOTEPORT}:localhost:{LOCALPORT} {hostname}
"""

# Run in background and disown
# Using bash so we can use & and disown
subprocess.Popen(
    ["bash", "-c", f"{ssh_command} & disown"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print("SSH reverse tunnel started in background.")

!pidof bash
!pidof ssh

In [ ]:
#!kill `pidof ssh`
!pidof ssh
#!ps -ef
!ss -tlnp
#!ls -l /root/.ssh/
#!cat /root/.ssh/colab_temp_key.pub
!ss -tunp


In [ ]:
!nvidia-smi
!df -h
